In [1]:
from pyspark.sql import SparkSession  
import getpass 
username = getpass 
spark = SparkSession.builder.config('spark.ui.port','0').\
        config('spark.sql.warehouse.dir',f"/user/itv022063/warehouse").\
        enableHiveSupport().\
        master('yarn').\
        getOrCreate()

In [2]:
loanSchema="""
            loan_id string,
            member_id string,
            loan_amnt float,
            funded_amnt float,
            term string,
            int_rate float,
            installment float,
            issue_d string,
            loan_status string,
            purpose string,
            title string
           """ 

In [3]:
loandf=spark.read.format('csv')\
.schema(loanSchema)\
.option("header","true") \
.load("/user/itv022063/Yash/raw/Loan")

In [4]:
loandf.printSchema()

root
 |-- loan_id: string (nullable = true)
 |-- member_id: string (nullable = true)
 |-- loan_amnt: float (nullable = true)
 |-- funded_amnt: float (nullable = true)
 |-- term: string (nullable = true)
 |-- int_rate: float (nullable = true)
 |-- installment: float (nullable = true)
 |-- issue_d: string (nullable = true)
 |-- loan_status: string (nullable = true)
 |-- purpose: string (nullable = true)
 |-- title: string (nullable = true)



In [5]:
loandf=loandf.withColumnRenamed("loan_amnt","loan_amount")\
      .withColumnRenamed("funded_amnt","funded_amount")\
      .withColumnRenamed("int_rate","interest_rate")\
      .withColumnRenamed("issue_d","issue_date")\
      .withColumnRenamed("term","loan_term_months")\
      .withColumnRenamed("installment","monthly_installment")\
      .withColumnRenamed("purpose","loan_purpose")

In [6]:
loandf.printSchema()

root
 |-- loan_id: string (nullable = true)
 |-- member_id: string (nullable = true)
 |-- loan_amount: float (nullable = true)
 |-- funded_amount: float (nullable = true)
 |-- loan_term_months: string (nullable = true)
 |-- interest_rate: float (nullable = true)
 |-- monthly_installment: float (nullable = true)
 |-- issue_date: string (nullable = true)
 |-- loan_status: string (nullable = true)
 |-- loan_purpose: string (nullable = true)
 |-- title: string (nullable = true)



In [7]:
from pyspark.sql.functions import *
loandf=loandf.withColumn("injest_date",current_timestamp())

In [8]:
columns_to_check = ["loan_amount", "funded_amount", "loan_term_months",
"interest_rate", "monthly_installment", "issue_date", "loan_status",
"loan_purpose"]

In [9]:
loandf=loandf.na.drop(subset = columns_to_check)

In [10]:
display(loandf)

loan_id,member_id,loan_amount,funded_amount,loan_term_months,interest_rate,monthly_installment,issue_date,loan_status,loan_purpose,title,injest_date
56633077,b59d80da191f5b573...,3000.0,3000.0,36 months,7.89,93.86,Aug-2015,Fully Paid,credit_card,Credit card refin...,2025-12-31 03:22:...
55927518,202d9f56ecb7c3bc9...,15600.0,15600.0,36 months,7.89,488.06,Aug-2015,Fully Paid,credit_card,Credit card refin...,2025-12-31 03:22:...
56473345,e5a140c0922b554b9...,20000.0,20000.0,36 months,9.17,637.58,Aug-2015,Fully Paid,debt_consolidation,Debt consolidation,2025-12-31 03:22:...
56463188,e12aefc548f750777...,11200.0,11200.0,60 months,21.99,309.27,Aug-2015,Fully Paid,home_improvement,Home improvement,2025-12-31 03:22:...
56473316,1b3a50d854fbbf97e...,16000.0,16000.0,60 months,20.99,432.77,Aug-2015,Charged Off,debt_consolidation,Debt consolidation,2025-12-31 03:22:...
56663266,1c4329e5f17697127...,20000.0,20000.0,60 months,13.33,458.45,Aug-2015,Charged Off,debt_consolidation,Debt consolidation,2025-12-31 03:22:...
56483027,5026c86ad983175eb...,10000.0,10000.0,36 months,12.69,335.45,Aug-2015,Fully Paid,other,Other,2025-12-31 03:22:...
56613385,9847d8c1e9d0b2084...,23400.0,23400.0,60 months,19.19,609.46,Aug-2015,Current,small_business,Business,2025-12-31 03:22:...
56643620,8340dbe1adea41fb4...,16000.0,16000.0,36 months,5.32,481.84,Jul-2015,Fully Paid,debt_consolidation,Debt consolidation,2025-12-31 03:22:...
56533114,d4de0de3ab7d79ad4...,25450.0,25450.0,36 months,27.31,1043.24,Aug-2015,Charged Off,debt_consolidation,Debt consolidation,2025-12-31 03:22:...


In [11]:
loan = (
    loandf
    .withColumn(
        "loan_term_months",
        (regexp_replace(col("loan_term_months"), "months", "")
         .cast("float") / 12)
        .cast("int")
    )
    .withColumnRenamed("loan_term_months", "loan_term_years")
)

In [28]:
loan.groupby("loan_purpose")\
    .agg(count("*").alias("total"))\
    .orderBy(col("total").desc())

loan_purpose,total
debt_consolidation,1277790
credit_card,516926
home_improvement,150440
other,139413
major_purchase,50429
medical,27481
small_business,24659
car,24009
vacation,15525
moving,15402


In [30]:
loan_purpose_lookup = ["debt_consolidation", "credit_card",
"home_improvement", "other", "major_purchase", "medical", "small_business",
"car", "vacation","moving", "house", "wedding", "renewable_energy",
"educational"]

In [31]:
loan_purpose_df=loan.withColumn("loan_purpose",when(col("loan_purpose").isin(loan_purpose_lookup),col("loan_purpose")).otherwise("other"))

In [32]:
loan_purpose_df

loan_id,member_id,loan_amount,funded_amount,loan_term_years,interest_rate,monthly_installment,issue_date,loan_status,loan_purpose,title,injest_date
56633077,b59d80da191f5b573...,3000.0,3000.0,3,7.89,93.86,Aug-2015,Fully Paid,credit_card,Credit card refin...,2025-12-31 03:39:...
55927518,202d9f56ecb7c3bc9...,15600.0,15600.0,3,7.89,488.06,Aug-2015,Fully Paid,credit_card,Credit card refin...,2025-12-31 03:39:...
56473345,e5a140c0922b554b9...,20000.0,20000.0,3,9.17,637.58,Aug-2015,Fully Paid,debt_consolidation,Debt consolidation,2025-12-31 03:39:...
56463188,e12aefc548f750777...,11200.0,11200.0,5,21.99,309.27,Aug-2015,Fully Paid,home_improvement,Home improvement,2025-12-31 03:39:...
56473316,1b3a50d854fbbf97e...,16000.0,16000.0,5,20.99,432.77,Aug-2015,Charged Off,debt_consolidation,Debt consolidation,2025-12-31 03:39:...
56663266,1c4329e5f17697127...,20000.0,20000.0,5,13.33,458.45,Aug-2015,Charged Off,debt_consolidation,Debt consolidation,2025-12-31 03:39:...
56483027,5026c86ad983175eb...,10000.0,10000.0,3,12.69,335.45,Aug-2015,Fully Paid,other,Other,2025-12-31 03:39:...
56613385,9847d8c1e9d0b2084...,23400.0,23400.0,5,19.19,609.46,Aug-2015,Current,small_business,Business,2025-12-31 03:39:...
56643620,8340dbe1adea41fb4...,16000.0,16000.0,3,5.32,481.84,Jul-2015,Fully Paid,debt_consolidation,Debt consolidation,2025-12-31 03:39:...
56533114,d4de0de3ab7d79ad4...,25450.0,25450.0,3,27.31,1043.24,Aug-2015,Charged Off,debt_consolidation,Debt consolidation,2025-12-31 03:39:...


In [33]:
loan_purpose_df.groupby("loan_purpose")\
    .agg(count("*").alias("total"))\
    .orderBy(col("total").desc())

loan_purpose,total
debt_consolidation,1277790
credit_card,516926
home_improvement,150440
other,139667
major_purchase,50429
medical,27481
small_business,24659
car,24009
vacation,15525
moving,15402


In [34]:
loan_purpose_df.write \
.format("parquet") \
.mode("overwrite") \
.option("path", "/user/itv022063/Yash/raw/loan_parquet") \
.save()

In [ ]:
loan_purpose_df.write \
.format("csv") \
.mode("overwrite") \
.option("path", "/user/itv022063/Yash/raw/loan_csv") \
.save()